In [1]:
# ==== S4d: WHICH CAMERA FEATURE VERSION CARRIES THE CAR'S POSITION ACROSS DRIVES? (no training, ~1 min)
import os
import numpy as np

WORK = r"C:\Users\abdel\OneDrive\Desktop\ODT"
if os.path.isdir(WORK):
    os.chdir(WORK)
D = "S32"
S = np.load(os.path.join(D, "S2_samples.npz"), allow_pickle=True)
rows, split, beam, power, seq = S["row"], S["split"].astype(str), S["beam"], S["power"].astype(np.float64), S["seq"]
F0 = np.load(os.path.join(D, "S4_cam_feats.npy"), mmap_mode="r")[0].astype(np.float32)   # originals (frames, 512, 12)
tr, va = np.flatnonzero(split == "train"), np.flatnonzero(split == "val")


def evaluate(scores, b, p, delta=5, K=3):
    rank = np.argsort(-scores, 1); err = rank[:, 0] - b
    Y = [1 - np.mean(np.min(np.minimum(np.abs(rank[:, :k] - b[:, None]) / delta, 1), 1)) for k in range(1, K + 1)]
    return dict(top1=100 * np.mean(err == 0), top3=100 * np.mean((rank[:, :3] == b[:, None]).any(1)),
                power=100 * np.mean(p[np.arange(len(b)), rank[:, 0]] / np.maximum(p.max(1), 1e-12)), dba=100 * np.mean(Y))


def knn(X, k=25):
    """cosine kNN: train samples -> validation samples"""
    X = X.reshape(len(X), -1); X = X - X[tr].mean(0); X = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
    sim = X[va] @ X[tr].T; nb = np.argsort(-sim, 1)[:, :k]
    sc = np.zeros((len(va), 64))
    for j in range(k):
        np.add.at(sc, (np.arange(len(va)), beam[tr][nb[:, j]]), 1.0 / (j + 1))
    return evaluate(sc, beam[va], power[va])


ft = F0[rows[:, -1]]                                               # frame t                      (S, 512, 12)
fm = F0[rows[:, -1]] - F0[rows[:, 0]]                              # motion: frame t - frame t-4   (causal)
dmean = {s: F0[np.unique(rows[seq == s])].mean(0) for s in np.unique(seq)}
fb = np.stack([F0[r[-1]] - dmean[s] for r, s in zip(rows, seq)])  # per-drive background removed (diagnostic only)
col = lambda X: np.abs(X).mean(1)                                  # 12 numbers: activity per image column

mc = np.bincount(beam[tr], minlength=64).argmax(); sc = np.zeros((len(va), 64)); sc[:, mc] = 1
print(f"{'validation drives, kNN train -> val':52s}{'top-1':>8s}{'top-3':>8s}{'power':>8s}{'DBA':>8s}")
for name, X in (("floor: always the most common beam", None),
                ("1 raw features, frame t (what S4b used)", ft),
                ("2 motion: frame t - frame t-4 (causal)", fm),
                ("3 per-drive background removed (diagnostic)", fb),
                ("2b motion, column activity only (12 numbers)", col(fm)),
                ("3b background removed, column activity only", col(fb))):
    r = evaluate(sc, beam[va], power[va]) if X is None else knn(X)
    print(f"{name:52s}{r['top1']:8.1f}{r['top3']:8.1f}{r['power']:8.1f}{r['dba']:8.1f}")
for name, X in (("motion", col(fm)), ("background removed", col(fb))):
    hot = X[va].argmax(1)
    print(f"corr(most active column, beam) on validation, {name}: {np.corrcoef(hot, beam[va])[0, 1]:+.3f}")

validation drives, kNN train -> val                    top-1   top-3   power     DBA
floor: always the most common beam                      18.6    20.9    81.8    43.8
1 raw features, frame t (what S4b used)                  3.5     9.7    70.2    14.7
2 motion: frame t - frame t-4 (causal)                   7.9    28.7    79.2    41.8
3 per-drive background removed (diagnostic)             11.0    26.0    78.5    35.2
2b motion, column activity only (12 numbers)             3.7    14.7    74.3    30.5
3b background removed, column activity only              6.2    18.4    74.9    33.1
corr(most active column, beam) on validation, motion: +0.272
corr(most active column, beam) on validation, background removed: +0.324
